In [ ]:
%matplotlib inline

# Notebook 3 — Beam–beam interaction and dynamic aperture

Install 6D beam–beam lenses at the four IPs and compare 200-turn survival with crab sextupoles on and off, then with both beam–beam and crab sextupoles off.

The opposing bunch is fixed (weak–strong model). Radiation and beamstrahlung emission are off.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xtrack as xt
import xfields as xf
from xfields.config_tools.beambeam_config_tools.config_tools import constant_charge_slicing_gaussian

### Beam parameters

Use the parameters from the [FCC Week 2025 presentation, slide 4](https://indico.cern.ch/event/1408515/contributions/6515086/attachments/3070936/5433965/FCCweek_Collimators_Impedance_Gibellieri_250520.pdf): $N=2.18\times10^{11}$, $\varepsilon_x=0.71$ nm rad, $\varepsilon_y=2.1$ pm rad, and the colliding bunch length $\sigma_z=15.2$ mm.

The full crossing angle is 30 mrad ([FCC-ee IR design](https://doi.org/10.1140/epjti/s40485-025-00117-3)). Use the supplied lattice's own IP beta functions (90 mm / 0.8 mm), rather than the published 110 mm / 0.7 mm. The colliding bunch length is a fixed input here.

In [ ]:
bunch_population = 2.18e11
gemitt_x = 0.71e-9  # m, geometric emittance
gemitt_y = 2.1e-12  # m, geometric emittance
sigma_z = 15.2e-3   # m, colliding bunch length
crossing_angle = 30e-3  # rad, full angle
n_slices = 201

### Load the nominal optics

Use 6D Twiss with the existing RF cavities. The small probe steps are needed to resolve the narrow vertical beam when beam–beam is enabled.

In [ ]:
env = xt.load('fccee_z.json')
env.vars.load('optics_nominal.json')
line = env.fccee_p_ring
line.configure_radiation(model=None)
ips = ['ipa', 'ipd', 'ipg', 'ipj']
crab_nominal = env['cs_frac']

beta_gamma = line.particle_ref.beta0[0] * line.particle_ref.gamma0[0]
twiss_kwargs = dict(
    method='6d', chrom=False,
    nemitt_x=gemitt_x * beta_gamma,
    nemitt_y=gemitt_y * beta_gamma,
    step_W_sigma=0.001,
    steps_R_matrix=dict(
        dx=1e-8, dpx=1e-9, dy=1e-10, dpy=1e-10,
        dzeta=1e-4, ddelta=1e-5,
    ),
)
tw0 = line.twiss(**twiss_kwargs)
tw0.rows[ips].cols['betx bety']

### Opposing beam and longitudinal slices

Reverse the Twiss direction to describe the opposing electron beam. Use equal-charge Gaussian slices, ordered in decreasing zeta.

In [ ]:
tw_other = tw0.reverse()
sigmas = tw_other.get_beam_covariance(gemitt_x=gemitt_x, gemitt_y=gemitt_y)

z_centroids, z_cuts, num_particles_per_slice = constant_charge_slicing_gaussian(
    N_part_tot=bunch_population, sigmaz=sigma_z, N_slices=n_slices,
)
z_centroids = z_centroids[::-1]
num_particles_per_slice = num_particles_per_slice[::-1]

### Install the lenses

The supplied code installs a lens at each IP. Xfields takes the half crossing angle. `on_bb` controls all four lenses; each lens represents the full opposing bunch.

In [ ]:
line.discard_tracker()
env['on_bb'] = 0
for ip in ips:
    bb_name = f'bb_{ip}'
    bb = xf.BeamBeamBiGaussian3D(
        phi=crossing_angle / 2,
        alpha=0,
        other_beam_q0=-line.particle_ref.q0,
        slices_other_beam_num_particles=num_particles_per_slice,
        slices_other_beam_zeta_center=z_centroids,
        slices_other_beam_Sigma_11=sigmas['Sigma11', ip],
        slices_other_beam_Sigma_12=sigmas['Sigma12', ip],
        slices_other_beam_Sigma_22=sigmas['Sigma22', ip],
        slices_other_beam_Sigma_33=sigmas['Sigma33', ip],
        slices_other_beam_Sigma_34=sigmas['Sigma34', ip],
        slices_other_beam_Sigma_44=sigmas['Sigma44', ip],
        flag_beamstrahlung=0,
    )
    line.insert(bb_name, bb, at=ip, with_progress=False)
    env.ref[bb_name].scale_strength = env.ref['on_bb']
line.build_tracker()

### Beam–beam tune shift

Compare the core tunes with the interaction off and on. Keep the beam–beam-on Twiss for the common launch coordinates used below.

In [ ]:
env['on_bb'] = 0
tw_off = line.twiss(**twiss_kwargs)
env['on_bb'] = 1
tw_bb = line.twiss(**twiss_kwargs)
print('BB off:', tw_off.qx, tw_off.qy)
print('BB on:', tw_bb.qx, tw_bb.qy)

### Add loss apertures

Place identical illustrative apertures near the downstream vertical-beta maximum at each IP: 8 horizontal and 6 vertical beam sizes from the original optics. These are teaching apertures, not FCC-ee hardware dimensions.

In [ ]:
for ip in ips:
    s_ip = tw0['s', ip]
    indices = np.flatnonzero((tw0.s > s_ip) & (tw0.s < s_ip + 20))
    i_ap = indices[np.argmax(tw0.bety[indices])]
    half_x = 8 * np.sqrt(gemitt_x * tw0.betx[i_ap])
    half_y = 6 * np.sqrt(gemitt_y * tw0.bety[i_ap])
    line.insert(
        f'loss_ap_{ip}',
        xt.LimitRect(min_x=-half_x, max_x=half_x,
                     min_y=-half_y, max_y=half_y),
        at=float(tw0.s[i_ap]), with_progress=False,
    )
    print(f'{ip}: half-apertures {half_x * 1e3:.2f} mm / {half_y * 1e3:.2f} mm')
line.build_tracker()

### Generate on-momentum particles

Use 16 radii from 0.5 to 8 sigmas and 10 angles in the first quadrant, with zero normalized transverse momenta and `zeta=delta=0`. Use the same beam–beam-on normalization and initial particles in all three cases.

In [ ]:
num_turns = 200
r_values = np.linspace(0.5, 8, 16)
theta_values = np.linspace(0, np.pi / 2, 10)
r_grid, theta_grid = np.meshgrid(r_values, theta_values)
Ax = (r_grid * np.cos(theta_grid)).ravel()
Ay = (r_grid * np.sin(theta_grid)).ravel()

particles_initial = line.build_particles(
    x_norm=..., px_norm=...,
    y_norm=..., py_norm=...,
    zeta=0, delta=0,
    gemitt_x=gemitt_x, gemitt_y=gemitt_y,
    particle_on_co=tw_bb.particle_on_co,
    W_matrix=tw_bb.W_matrix[0],
)

### Beam–beam on, crab sextupoles on

In [ ]:
env['on_bb'] = ...
env['cs_frac'] = ...
p_on = particles_initial.copy()
line.track(p_on, num_turns=num_turns, time=True, with_progress=10)
print('Tracking time:', line.time_last_track)  # s

### Beam–beam on, crab sextupoles off

In [ ]:
env['on_bb'] = ...
env['cs_frac'] = ...
p_crab_off = particles_initial.copy()
line.track(p_crab_off, num_turns=num_turns, time=True, with_progress=10)
print('Tracking time:', line.time_last_track)  # s

### Beam–beam off, crab sextupoles off

In [ ]:
env['on_bb'] = ...
env['cs_frac'] = ...
p_both_off = particles_initial.copy()
line.track(p_both_off, num_turns=num_turns, time=True, with_progress=10)
print('Tracking time:', line.time_last_track)  # s

### Compare survival

Restore particle-ID order before connecting survival to the initial grid. Small C0 dots survived; larger red dots were lost.

In [ ]:
p_on.sort(interleave_lost_particles=True)
p_crab_off.sort(interleave_lost_particles=True)
p_both_off.sort(interleave_lost_particles=True)
survived_on = p_on.state > 0
survived_crab_off = p_crab_off.state > 0
survived_both_off = p_both_off.state > 0
print('Lost, BB on / crab on:', np.count_nonzero(~survived_on))
print('Lost, BB on / crab off:', np.count_nonzero(~survived_crab_off))
print('Lost, BB off / crab off:', np.count_nonzero(~survived_both_off))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True, sharey=True, layout='constrained')

axes[0].scatter(Ax[survived_on], Ay[survived_on], color='C0', s=15, label='Survived')
axes[0].scatter(Ax[~survived_on], Ay[~survived_on], color='red', s=55, label='Lost')
axes[0].set_title('BB on, crab on')
axes[0].set_xlabel(r'$A_x$ [$\sigma_x$]')
axes[0].set_ylabel(r'$A_y$ [$\sigma_y$]')
axes[0].set_aspect('equal')
axes[0].legend()

axes[1].scatter(Ax[survived_crab_off], Ay[survived_crab_off], color='C0', s=15)
axes[1].scatter(Ax[~survived_crab_off], Ay[~survived_crab_off], color='red', s=55)
axes[1].set_title('BB on, crab off')
axes[1].set_xlabel(r'$A_x$ [$\sigma_x$]')
axes[1].set_aspect('equal')

axes[2].scatter(Ax[survived_both_off], Ay[survived_both_off], color='C0', s=15)
axes[2].scatter(Ax[~survived_both_off], Ay[~survived_both_off], color='red', s=55)
axes[2].set_title('BB off, crab off')
axes[2].set_xlabel(r'$A_x$ [$\sigma_x$]')
axes[2].set_aspect('equal')

How does switching off the crab sextupoles affect survival with beam–beam on? What changes when beam–beam is also switched off?

This is a coarse finite-turn scan with added apertures. Particles start on momentum; RF and beam–beam kicks can subsequently change their momentum.